# Run Qwen on AMD MI300X for Spice Wizard

This notebook runs `Qwen/Qwen2.5-Coder-7B-Instruct` on an AMD MI300X through ROCm, generates a constrained LTspice netlist adaptation, and saves the candidate for local verification by Spice Wizard.

**Demo pipeline:**

```text
AMD MI300X: Qwen 7B generates a template adaptation
                         │ AD8092_gain5.net
                         ▼
Mac: Spice Wizard runs LTspice, measures the result, and reports PASS or FAIL
```

Run cells from top to bottom. Keep the kernel alive while recording `rocm-smi` evidence. This notebook uses a manual handoff, so it does not expose a public server or contain an API key.

## Step 0 — Install notebook dependencies

The AMD environment supplies ROCm-compatible PyTorch. Install only the user-space packages needed to download, load, and generate from Qwen.

In [ ]:
import os

# Prefer the standard Hugging Face download path for predictable cloud behavior.
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DISABLE_XET"] = "1"

%pip -q install -U huggingface_hub transformers accelerate safetensors ipywidgets
print("Notebook dependencies installed; standard Hugging Face download path selected.")

## Step 1 — Download and load Qwen 7B on the MI300X

This is the demo-quality model for the hackathon. The weights persist under `~/models/`, so repeat sessions reuse the local download. The cell fails early if the ROCm device is unavailable.

In [ ]:
import os
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
MODEL_DIR = Path.home() / "models" / "qwen2.5-coder-7b-instruct"

if not torch.cuda.is_available():
    raise RuntimeError("No ROCm/CUDA device is available. Select the MI300X runtime before continuing.")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Loading {MODEL_NAME} from {MODEL_DIR} (downloads on first run)...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    cache_dir=str(MODEL_DIR),
    trust_remote_code=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    cache_dir=str(MODEL_DIR),
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("Qwen is ready.")

## Step 2 — Prove inference on the MI300X

**Open a Terminal and run `rocm-smi` while this cell runs** — GPU% / VRAM% jumping
is your AMD-compute proof shot. Screenshot it for the README and demo video.

In [ ]:
messages = [{"role": "user",
             "content": "In this LTspice line, change the gain from 2 to 5 by editing only R2: "
                        "'R2 OUT N001 1K'. Reply with only the new line."}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
print(tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))

In [ ]:
import re

def extract_netlist(text):
    m = re.search(r"```(?:spice)?\s*\n(.*?)```", text, re.DOTALL)
    body = m.group(1) if m else text
    lines = body.splitlines()
    end = max((i for i, line in enumerate(lines) if line.strip().lower() in (".end", ".ends")), default=None)
    if end is not None:
        lines = lines[:end + 1]
    return "\n".join(lines).strip() + "\n"

def generate_netlist(template, ic_name, spec_text, max_new_tokens=800):
    prompt = f"""Below is a KNOWN-WORKING LTspice netlist template for the {ic_name}.
Modify ONLY the passive component values (resistors, capacitors, source amplitudes/frequencies)
needed to meet the NEW SPEC. Do NOT change the {ic_name} subcircuit call line's pin order or node
names. Do NOT change the .lib reference. Output the complete modified netlist and nothing else.

TEMPLATE:
{template}

NEW SPEC:
{spec_text}
"""
    messages = [
        {"role": "system", "content": "You are an expert analog applications engineer who writes LTspice netlists."},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    raw = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return extract_netlist(raw)

AD8092_TEMPLATE = r'''* AD8092 non-inverting amp template
X§U1 IN N001 +V -V OUT AD8092
Vin IN 0 SINE(0 0.1 1Meg)
R1 N001 0 1K
R2 OUT N001 1K
Rload OUT 0 2K
V1 +V 0 5
V2 -V 0 -5
.tran 1u
.lib ADI.lib
.backanno
.end
'''

result = generate_netlist(
    AD8092_TEMPLATE, "AD8092",
    "Non-inverting amplifier, closed-loop gain of 5 V/V (~13.98 dB), +/-5V supplies, "
    "use a 100mV amplitude 1MHz sine test signal to stay within the part's linear range."
)
print(result)

# Download this candidate with the Jupyter file browser.
with open("AD8092_gain5.net", "w", encoding="utf-8") as candidate_file:
    candidate_file.write(result)
print("\nSaved AD8092_gain5.net")

## Step 3 — Transfer the candidate for LTspice verification on macOS

1. Download `AD8092_gain5.net` from this notebook environment.
2. Copy it to a local Spice Wizard checkout, for example `outputs/AD8092_gain5.net`.
3. On the Mac, verify it with LTspice:

```bash
python generate_verify.py AD8092 \
  --spec "non-inverting gain of 5 V/V, +/-5 V supplies, 100 mV at 1 MHz" \
  --metric gain_db=13.98:1.0 \
  --freq 1e6 \
  --candidate outputs/AD8092_gain5.net \
  --source amd_mi300x_manual
```

The verifier records the candidate, launches LTspice, measures the waveform, and writes a numeric PASS/FAIL report. No tunnel, web server, or notebook secret is required.